# Download the NYC taxi datasets
Download NYC TLC trip record data (https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). Files will be saved on `/Volumes/dbx_joshdevph_dev/raw/vlm` volume. Files to be downloaded are as follows. 
- Taxi Zone Lookup Table (CSV)
- Taxi Zone Shapefile (PARQUET)
- Yellow Taxi Trip Records (PARQUET) (2024 to current year)

In [0]:
%pip install geopandas fsspec --quiet

## Download raw TLC Trip Record Data from source

In [0]:
import datetime as dt
import zipfile
import requests
from pathlib import Path

# Define the volume path
volume_path = "/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/"

# Create the directory if it doesn't exist
Path(volume_path).mkdir(exist_ok=True, parents=True)

# Define the URLs
urls = [
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"]

# Construct the download URLs for the yellow taxi datasets.
# Adjust starting year as needed
starting_year = 2024
for y in range(starting_year, dt.datetime.now().year + 1):
    for m in range(1, 13):
        urls.append(f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet")

# Download each file
for url in urls:
    filename = url.split("/")[-1]
    filepath = Path(volume_path) / filename
    
    print(f"Downloading {filename}...")
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"Successfully downloaded {filename} to {filepath}")
    else:
        # Skip files that threw an error on download
        print(f"Failed to download {filename}. Status code: {response.status_code}")

print("\nDownload complete!")

## Uncompress zipped shape files

In [0]:
# Uncompress the zip file 
try:
    with zipfile.ZipFile("/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/taxi_zones.zip", "r") as zip_ref:
        zip_ref.extractall(volume_path)
    print("Successfully uncompressed the files.")

except Exception as e:
    print(f"Failed to uncompress the files. {e}")